<a href="https://colab.research.google.com/github/hjx-zju/blog/blob/master/colabs/genception_inference_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

# GenCeption Inference Demo

This notebook demonstrates how to run inference with the [GenCeption](https://arxiv.org/abs/2607.09024) model. GenCeption is a generalist video foundation model built on the WAN 2.1 diffusion transformer architecture, fine-tuned for multi-modal video tasks such as:

- **Depth estimation** from RGB video
- **Surface normal estimation**
- **Referring video object segmentation (RefVOS)**
- **Arbitrary visual understanding tasks**

Inference is a **single forward pass**: the model encodes the input video via the VAE, runs one transformer step, and decodes the predicted latents back to pixel space. Text prompt embeddings are precomputed, allowing fast, memory-efficient inference without loading the 22 GB UMT5-XXL text encoder.

Both the **1.3B** and **14B** models are supported — set `MODEL_SIZE` below.

In [1]:
# @title Installation {form-width: "10%"}

!pip install -q jax[cuda12] flax==0.12.9 mediapy safetensors
!pip install -q git+https://github.com/google-deepmind/representations4d.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.1/532.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 95.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 56.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.3/609.3 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# @title Imports {form-width: "10%"}

import os
import numpy as np
import jax
import jax.numpy as jnp
import mediapy as media

print(f"JAX devices: {jax.devices()}")
print(f"JAX version: {jax.__version__}")

JAX devices: [CudaDevice(id=0)]
JAX version: 0.11.1


In [3]:
# @title Download checkpoints and prompt embeddings {form-width: "10%"}

MODEL_SIZE = '1.3b'  # @param ['1.3b', '14b']

CHECKPOINT_DIR = f'content/genception_{MODEL_SIZE}'
EMBEDDINGS_DIR = 'content/prompt_embeddings'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
os.makedirs(f'{EMBEDDINGS_DIR}/refvos/davis', exist_ok=True)
os.makedirs(f'{EMBEDDINGS_DIR}/refvos/mevis', exist_ok=True)

BASE_CKPT_URL = 'https://storage.googleapis.com/representations4d/checkpoints/genception'
BASE_EMBEDS_URL = 'https://storage.googleapis.com/representations4d/assets/genception_prompt_embeddings'

# 1. Download transformer weights & config
!wget -q -nc {BASE_CKPT_URL}/genception_{MODEL_SIZE}_transformer.npz -P {CHECKPOINT_DIR}
!wget -q -nc {BASE_CKPT_URL}/genception_{MODEL_SIZE}_config.json -P {CHECKPOINT_DIR}

# 2. Download VAE
os.makedirs(f'{CHECKPOINT_DIR}/vae', exist_ok=True)
!wget -q -nc {BASE_CKPT_URL}/vae/config.json -P {CHECKPOINT_DIR}/vae/
!wget -q -nc {BASE_CKPT_URL}/vae/diffusion_pytorch_model.safetensors -P {CHECKPOINT_DIR}/vae/

# 3. Download precomputed prompt embeddings (depth, surface normal, negative, RefVOS)
for name in ['depth.npy', 'normals.npy', 'negative.npy']:
    !wget -q -nc {BASE_EMBEDS_URL}/{name} -P {EMBEDDINGS_DIR}

# Default referring expression embedding ("a man.") for the demo video
!wget -q -nc {BASE_EMBEDS_URL}/refvos/davis/bike-packing_cap_a_man..npy -P {EMBEDDINGS_DIR}/refvos/davis/

print(f"Checkpoints ready at {CHECKPOINT_DIR}")
print(f"Embeddings ready at {EMBEDDINGS_DIR}")

Checkpoints ready at content/genception_1.3b
Embeddings ready at content/prompt_embeddings


In [6]:
# @title Load model and embeddings {form-width: "10%"}

from representations4d.genception import GenCeptionPipeline

pipe = GenCeptionPipeline.from_pretrained(CHECKPOINT_DIR, model_size=MODEL_SIZE)
print("GenCeption pipeline loaded successfully!")

# Load precomputed prompt embeddings
depth_embeds = np.load(f'{EMBEDDINGS_DIR}/depth.npy')
surface_normal_embeds = np.load(f'{EMBEDDINGS_DIR}/normals.npy')
ref_man_embeds = np.load(f'{EMBEDDINGS_DIR}/refvos/davis/bike-packing_cap_a_man..npy')

print(f"Loaded prompt embeddings: shape={depth_embeds.shape}, dtype={depth_embeds.dtype}")

GenCeption pipeline loaded successfully!
Loaded prompt embeddings: shape=(1, 226, 4096), dtype=float32


## Load an example video

We'll download a short example clip and display it. You can replace this with your own video.

In [7]:
# @title Load example video {form-width: "10%"}

# Download example video (man in warehouse clip)
!wget -q -nc https://storage.googleapis.com/representations4d/assets/man_in_warehouse.mp4 -O content/man_in_warehouse.mp4

# Load and display the video
video = media.read_video('content/man_in_warehouse.mp4')

print(f"Input video shape: {video.shape} (T={video.shape[0]}, H={video.shape[1]}, W={video.shape[2]})")
media.show_video(video, fps=25, title='Input video')

Input video shape: (358, 480, 832, 3) (T=358, H=480, W=832)


Input video This browser does not support the video tag.


## Depth Estimation

GenCeption estimates depth from RGB video using the precomputed prompt embedding for `"Input: RGB. Output: Depth."`. Inference runs a single forward pass through the transformer.

In [8]:
# @title Run depth estimation {form-width: "10%"}

result = pipe(
    prompt_embeds=depth_embeds,
    input_video=video,
)

depth_video = result['video']
print(f"Output shape: {depth_video.shape}")

# Display comparison
media.show_video(video, fps=25, title='Input RGB')
media.show_video(depth_video, fps=25, title='Predicted Depth')

/usr/local/lib/python3.13/dist-packages/jax/_src/numpy/lax_numpy.py:5932: UserWarning: Explicitly requested dtype float64 requested in arange is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  return _arange(start, stop=stop, step=step, dtype=dtype,


JaxRuntimeError: INTERNAL: SDPA FP16/BF16 requires SM80 (Ampere) or newer architecture
in external/xla+/xla/stream_executor/cuda/cuda_dnn.cc(6893): 'graph_.validate()' 
	

Suspected Python Code Location:
  /usr/local/lib/python3.13/dist-packages/representations4d/genception/attention.py:109:16 [_apply_flash_attention]
  /usr/local/lib/python3.13/dist-packages/representations4d/genception/attention.py:376:13 [WanAttention._run_attention]
  /usr/local/lib/python3.13/dist-packages/representations4d/genception/attention.py:446:12 [WanAttention.__call__]
  /usr/local/lib/python3.13/dist-packages/representations4d/genception/model.py:664:20 [WanTransformerBlock.__call__]
  /usr/local/lib/python3.13/dist-packages/representations4d/genception/model.py:1139:22 [WanModel.__call__.<locals>.scan_fn]
  /usr/local/lib/python3.13/dist-packages/flax/nnx/transforms/iteration.py:1320:10 [ScanFn.__call__]
  /usr/local/lib/python3.13/dist-packages/flax/nnx/transforms/iteration.py:1813:26 [_graph_updates_s

## Surface Normal Estimation

The model predicts surface normals by supplying the embedding for `"Input: RGB. Output: Surface Normal."`. Surface normals are encoded as 3-channel RGB maps corresponding to the surface orientation vectors.

In [ ]:
# @title Run surface normal estimation {form-width: "10%"}

result = pipe(
    prompt_embeds=surface_normal_embeds,
    input_video=video,
)

normal_video = result['video']
print(f"Output shape: {normal_video.shape}")

media.show_video(video, fps=25, title='Input RGB')
media.show_video(normal_video, fps=25, title='Predicted Surface Normals')

## Referring Expression Video Segmentation (RefVOS)

GenCeption can segment specific objects in video based on natural-language referring expressions. Here we use the precomputed embedding corresponding to the referring expression **"a man."** (`bike-packing_cap_a_man..npy` from the DAVIS dataset), which segments the person in our demo warehouse video.

Full sets of precomputed embeddings for the DAVIS (`davis`) and MeViS (`mevis`) datasets are available at `https://storage.googleapis.com/representations4d/assets/genception_prompt_embeddings/refvos/`.

In [ ]:
# @title Run referring expression segmentation {form-width: "10%"}

# Using the precomputed embedding for referring expression: "a man."
result = pipe(
    prompt_embeds=ref_man_embeds,
    input_video=video,
)

seg_video = result['video']
print(f"Output shape: {seg_video.shape}")

media.show_video(video, fps=25, title='Input RGB')
media.show_video(seg_video, fps=25, title='Segmentation: "a man."')